###PHASE 2 : SPARK STRUCTURED STREAMING PIPELINE

Step 1 - Verifying our clean data is readable

In [0]:
df = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/raw_data/clean")
print(f"Row count: {df.count()}")
df.printSchema()

Row count: 49962
root
 |-- step: double (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: double (nullable = true)



Before building the streaming pipeline, we first verify that the cleaned dataset from Phase 1 is correctly stored in Delta Lake and accessible. We load it in standard batch mode and confirm the row count (49,962 rows) and schema, ensuring all 11 columns have the correct data types before switching to streaming mode

Step 2 — Define the schema explicitly

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

schema = StructType([
    StructField("step", DoubleType(), True),
    StructField("type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("nameOrig", StringType(), True),
    StructField("oldbalanceOrg", DoubleType(), True),
    StructField("newbalanceOrig", DoubleType(), True),
    StructField("nameDest", StringType(), True),
    StructField("oldbalanceDest", DoubleType(), True),
    StructField("newbalanceDest", DoubleType(), True),
    StructField("isFraud", IntegerType(), True),
    StructField("isFlaggedFraud", DoubleType(), True)
])

print("Schema defined successfully.")

Schema defined successfully.


Spark Structured Streaming requires the schema to be defined manually before reading data. Unlike batch mode, streaming cannot scan the full dataset upfront to infer column types. We define all 11 columns with their exact data types matching the cleaned dataset: doubles for numeric fields, strings for names and transaction type, and integer for the fraud label.

Step 3 — Reading the data as a stream

In [0]:
streaming_df = (
    spark.readStream
    .format("delta")
    .option("maxFilesPerTrigger", 1)
    .load("dbfs:/Volumes/workspace/default/raw_data/clean")
)

print("Is streaming:", streaming_df.isStreaming)

Is streaming: True


We open the cleaned Delta table as a Spark Structured Stream. Delta Lake stores schema information internally, so no manual schema definition is needed at read time. The maxFilesPerTrigger option is set to 1, simulating transactions arriving in small batches over time. We confirm the DataFrame is in streaming mode using isStreaming, which returns True.

Step 4 — Adding feature columns

In [0]:
from pyspark.sql.functions import col, when

featured_df = streaming_df.withColumn(
    "is_high_risk_type",
    when(col("type").isin("TRANSFER", "CASH_OUT"), 1).otherwise(0)
).withColumn(
    "balance_diff",
    col("oldbalanceOrg") - col("newbalanceOrig")
)

print("Features added. Columns:", featured_df.columns)

Features added. Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'is_high_risk_type', 'balance_diff']


Two feature columns are added to the stream. The first, is_high_risk_type, flags transactions of type TRANSFER or CASH_OUT — the only two types where fraud appears in this dataset. The second, balance_diff, captures the drop in the sender's balance, which serves as a behavioral fraud signal. These features will feed directly into the MLlib model in Phase 3.



Step 5 — Writing the stream to a sink

In [0]:
from pyspark.sql.streaming import DataStreamWriter

query = (
    featured_df.writeStream
    .format("delta")
    .option("checkpointLocation", "dbfs:/Volumes/workspace/default/raw_data/checkpoint")
    .outputMode("append")
    .trigger(availableNow=True)
    .start("dbfs:/Volumes/workspace/default/raw_data/streaming_output")
)

query.awaitTermination()

Due to a cluster-type restriction in Databricks Community Edition, the availableNow trigger is used instead of a continuous processing trigger. This mode processes all currently available micro-batches and then terminates cleanly, fully demonstrating the streaming pipeline behavior while remaining compatible with the Community Edition environment.

Step 6 — Verifying the output

In [0]:
output_df = spark.read.format("delta").load("dbfs:/Volumes/workspace/default/raw_data/streaming_output")

print(f"Total rows written: {output_df.count()}")
print(f"Columns: {output_df.columns}")
output_df.select("type", "amount", "is_high_risk_type", "balance_diff", "isFraud").show(5)

Total rows written: 49962
Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud', 'is_high_risk_type', 'balance_diff']
+--------+---------+-----------------+------------+-------+
|    type|   amount|is_high_risk_type|balance_diff|isFraud|
+--------+---------+-----------------+------------+-------+
|CASH_OUT|118326.41|                1|   118326.41|      0|
| PAYMENT| 17068.83|                0|    17068.83|      0|
|CASH_OUT|369264.57|                1|    124268.0|      0|
| PAYMENT| 10039.08|                0|         0.0|      0|
| PAYMENT|  7639.89|                0|         0.0|      0|
+--------+---------+-----------------+------------+-------+
only showing top 5 rows


To verify the pipeline completed successfully, the output Delta table is read back in batch mode. We confirm the row count, the presence of the two engineered feature columns, and inspect a sample of rows to validate the data looks correct end-to-end.